In [ ]:
import ee

import geemap
import os
from google.colab import files
import geopandas as gpd
import json
import numpy as np
from glob import glob
from google.colab import drive
import matplotlib.pyplot as plt

# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='gee-tutorial-470209')

# Define the desired directory path
target_directory = '/content/Aoi'

uploaded = files.upload(target_directory)

In [ ]:
# Load GeoJSON
with open("/content/Aoi/AOI_Jamuna_DS_F.geojson") as f:
    geojson_data = json.load(f)

# Convert to EE Geometry or FeatureCollection
roi = ee.FeatureCollection(geojson_data)

In [ ]:
# Scale Landsat-8 Collection 2 Level-2 Surface Reflectance
def scale_landsat8(img):
    sr = img.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'])
    scaled = sr.multiply(0.0000275).add(-0.2)
    img = img.addBands(scaled, overwrite=True)
    return img

def get_ls8_images(year):
    # Filter collection for a specific year
    col = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(roi)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filter(ee.Filter.lt("CLOUD_COVER", 5))
        .map(scale_landsat8)
    )

    size = col.size().getInfo()
    print(f"📌 Year {year}: Found {size} images")

    if size == 0:
        return []

    # Convert to Python list of ee.Image
    img_list = col.toList(size)

    results = []

    for i in range(size):
        img = ee.Image(img_list.get(i))

        # Extract date
        date = ee.Date(img.get("system:time_start")).format("yyyyMMdd").getInfo()

        # Custom name
        name = f"LS08_{date}"

        # Select RGB Bands (False Color SWIR1, NIR, Red)
        rgb = img.select(['SR_B5', 'SR_B4', 'SR_B3'])

        # Attach metadata
        rgb = rgb.set({
            "name": name,
            "date": date,
            "year": year,
            "source": "LS8"
        })

        print(f"  → Added image: {name}")

        results.append(rgb)

    return results


# -----------------------------------------------------
# Process years
ls8_color_images = []

for year in range(2024, 2025):  # Landsat 8 available from 2013
    imgs = get_ls8_images(year)
    ls8_color_images.extend(imgs)


print("Total LS8 images collected:", len(ls8_color_images))



📌 Year 2024: Found 19 images
  → Added image: LS08_20240309
  → Added image: LS08_20241104
  → Added image: LS08_20240309
  → Added image: LS08_20240426
  → Added image: LS08_20241104
  → Added image: LS08_20241120
  → Added image: LS08_20241206
  → Added image: LS08_20241222
  → Added image: LS08_20240128
  → Added image: LS08_20240229
  → Added image: LS08_20240401
  → Added image: LS08_20240503
  → Added image: LS08_20241111
  → Added image: LS08_20241127
  → Added image: LS08_20241229
  → Added image: LS08_20240229
  → Added image: LS08_20240401
  → Added image: LS08_20240503
  → Added image: LS08_20241229
Total LS8 images collected: 19


In [ ]:
import geemap
import ipywidgets as widgets
from IPython.display import display

# Create date → image dictionary
date_options = []
date_to_image = {}

for img in ls8_color_images:   # <-- must contain all LS8 images
    date = img.get("date").getInfo()      # '20240212'
    name = img.get("name").getInfo()      # 'LS08_20240212'

    date_options.append(name)
    date_to_image[name] = img

print("Total images:", len(date_options))
print(date_options[:10])  # show first 10



# Create the map
Map = geemap.Map()
Map.centerObject(roi, 10)

# Dropdown widget (each item = LS08_YYYYMMDD)
dropdown = widgets.Dropdown(
    options=date_options,
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Update function
def on_date_change(change):
    Map.layers = [Map.layers[0]]  # keep only base layer

    name = change['new']
    rgb_image = date_to_image[name]

    # Add the selected Landsat RGB image
    Map.addLayer(
        rgb_image,
        {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min': 0, 'max': 0.3},
        f"Image {name}"
    )

dropdown.observe(on_date_change, names='value')

# Display
display(dropdown)
display(Map)

# Initialize with the first image
dropdown.value = date_options[0]


Total images: 19
['LS08_20240309', 'LS08_20241104', 'LS08_20240309', 'LS08_20240426', 'LS08_20241104', 'LS08_20241120', 'LS08_20241206', 'LS08_20241222', 'LS08_20240128', 'LS08_20240229']


Dropdown(description='Select Image:', layout=Layout(width='300px'), options=('LS08_20240309', 'LS08_20241104',…

Map(center=[24.018155540858295, 89.73023370217629], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Sentinel-2 Surface Reflectance scaling
def scale_s2(img):
    # Sentinel-2 Level-2A scale factor 0.0001
    scaled = img.select(['B2','B3','B4','B8']).multiply(0.0001)
    return img.addBands(scaled, overwrite=True)

def get_s2_images(year):
    # Filter S2 collection
    col = (
        ee.ImageCollection("COPERNICUS/S2_SR")
        .filterBounds(roi)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
        .map(scale_s2)
    )

    size = col.size().getInfo()
    print(f"📌 Year {year}: Found {size} Sentinel-2 images")

    if size == 0:
        return []

    # Convert to Python list
    img_list = col.toList(size)
    results = []

    for i in range(size):
        img = ee.Image(img_list.get(i))

        # Extract the date
        date = ee.Date(img.get("system:time_start")).format("yyyyMMdd").getInfo()

        # Custom image name
        name = f"S2_{date}"

        # Select RGB (NIR, Red, Green) → B8, B4, B3
        rgb = img.select(['B8', 'B4', 'B3'])

        # Add metadata
        rgb = rgb.set({
            "name": name,
            "date": date,
            "year": year,
            "source": "S2"
        })

        print(f"  → Added image: {name}")

        results.append(rgb)

    return results


# -----------------------------------------------------
# Process multiple years
s2_images = []

for year in range(2024, 2025):  # Good S2 SR availability after 2017
    imgs = get_s2_images(year)
    s2_images.extend(imgs)



📌 Year 2024: Found 35 Sentinel-2 images
  → Added image: S2_20240104
  → Added image: S2_20240106
  → Added image: S2_20240119
  → Added image: S2_20240126
  → Added image: S2_20240203
  → Added image: S2_20240208
  → Added image: S2_20240210
  → Added image: S2_20240218
  → Added image: S2_20240228
  → Added image: S2_20240301
  → Added image: S2_20240306
  → Added image: S2_20240309
  → Added image: S2_20240311
  → Added image: S2_20240316
  → Added image: S2_20240326
  → Added image: S2_20240413
  → Added image: S2_20240415
  → Added image: S2_20240420
  → Added image: S2_20240430
  → Added image: S2_20240503
  → Added image: S2_20241022
  → Added image: S2_20241109
  → Added image: S2_20241111
  → Added image: S2_20241114
  → Added image: S2_20241116
  → Added image: S2_20241119
  → Added image: S2_20241121
  → Added image: S2_20241204
  → Added image: S2_20241206
  → Added image: S2_20241214
  → Added image: S2_20241214
  → Added image: S2_20241216
  → Added image: S2_20241224
  →

In [ ]:
import geemap
import ipywidgets as widgets
from IPython.display import display

# Create date → image dictionary
date_options = []
date_to_image = {}

for img in s2_images:   # <-- must contain all LS8 images
    date = img.get("date").getInfo()      # '20240212'
    name = img.get("name").getInfo()      # 'S2_20240212'

    date_options.append(name)
    date_to_image[name] = img

print("Total images:", len(date_options))
print(date_options[:10])  # show first 10



# Create the map
Map = geemap.Map()
Map.centerObject(roi, 10)

# Dropdown widget (each item = LS08_YYYYMMDD)
dropdown = widgets.Dropdown(
    options=date_options,
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Update function
def on_date_change(change):
    Map.layers = [Map.layers[0]]  # keep only base layer

    name = change['new']
    rgb_image = date_to_image[name]

    # Add the selected Landsat RGB image
    Map.addLayer(
        rgb_image,
        {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 0.3},
        f"Image {name}"
    )

dropdown.observe(on_date_change, names='value')

# Display
display(dropdown)
display(Map)

# Initialize with the first image
dropdown.value = date_options[0]


Total images: 35
['S2_20240104', 'S2_20240106', 'S2_20240119', 'S2_20240126', 'S2_20240203', 'S2_20240208', 'S2_20240210', 'S2_20240218', 'S2_20240228', 'S2_20240301']


Dropdown(description='Select Image:', layout=Layout(width='300px'), options=('S2_20240104', 'S2_20240106', 'S2…

Map(center=[24.018155540858295, 89.73023370217629], controls=(WidgetControl(options=['position', 'transparent_…